In [1]:
import torch 
import torch.nn as nn
import torch.optim as optim

In [2]:
torch.manual_seed(11037)

In [3]:
X = torch.tensor([[1, 2, 3, 4],
                  [5, 6, 7, 8],
                  [10, 11, 12, 13],
                  [20, 21, 22, 23]], dtype=torch.float32)

Y = torch.tensor([[5], [9], [14], [24]], dtype=torch.float32)

In [4]:
X = X.unsqueeze(-1)
X

tensor([[[ 1.],
         [ 2.],
         [ 3.],
         [ 4.]],

        [[ 5.],
         [ 6.],
         [ 7.],
         [ 8.]],

        [[10.],
         [11.],
         [12.],
         [13.]],

        [[20.],
         [21.],
         [22.],
         [23.]]])

In [5]:
class RNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=64, hidden_size=128, *args, **kwargs):
        super().__init__(*args, **kwargs)

        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        self.rnn = nn.RNN(input_size=embedding_dim, hidden_size=hidden_size, batch_first=True)

        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        output, hidden = self.rnn(x)
        logits = self.fc(output)
        return logits, hidden

In [6]:
import requests

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

response = requests.get(url, timeout=30)
response.raise_for_status()

text = response.text

print("Total chars:", len(text))
print(text[:200])

Total chars: 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


In [7]:
chars = sorted(set(text))
chars

['\n',
 ' ',
 '!',
 '$',
 '&',
 "'",
 ',',
 '-',
 '.',
 '3',
 ':',
 ';',
 '?',
 'A',
 'B',
 'C',
 'D',
 'E',
 'F',
 'G',
 'H',
 'I',
 'J',
 'K',
 'L',
 'M',
 'N',
 'O',
 'P',
 'Q',
 'R',
 'S',
 'T',
 'U',
 'V',
 'W',
 'X',
 'Y',
 'Z',
 'a',
 'b',
 'c',
 'd',
 'e',
 'f',
 'g',
 'h',
 'i',
 'j',
 'k',
 'l',
 'm',
 'n',
 'o',
 'p',
 'q',
 'r',
 's',
 't',
 'u',
 'v',
 'w',
 'x',
 'y',
 'z']

In [8]:
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

In [9]:
stoi

{'\n': 0,
 ' ': 1,
 '!': 2,
 '$': 3,
 '&': 4,
 "'": 5,
 ',': 6,
 '-': 7,
 '.': 8,
 '3': 9,
 ':': 10,
 ';': 11,
 '?': 12,
 'A': 13,
 'B': 14,
 'C': 15,
 'D': 16,
 'E': 17,
 'F': 18,
 'G': 19,
 'H': 20,
 'I': 21,
 'J': 22,
 'K': 23,
 'L': 24,
 'M': 25,
 'N': 26,
 'O': 27,
 'P': 28,
 'Q': 29,
 'R': 30,
 'S': 31,
 'T': 32,
 'U': 33,
 'V': 34,
 'W': 35,
 'X': 36,
 'Y': 37,
 'Z': 38,
 'a': 39,
 'b': 40,
 'c': 41,
 'd': 42,
 'e': 43,
 'f': 44,
 'g': 45,
 'h': 46,
 'i': 47,
 'j': 48,
 'k': 49,
 'l': 50,
 'm': 51,
 'n': 52,
 'o': 53,
 'p': 54,
 'q': 55,
 'r': 56,
 's': 57,
 't': 58,
 'u': 59,
 'v': 60,
 'w': 61,
 'x': 62,
 'y': 63,
 'z': 64}

In [10]:
itos

{0: '\n',
 1: ' ',
 2: '!',
 3: '$',
 4: '&',
 5: "'",
 6: ',',
 7: '-',
 8: '.',
 9: '3',
 10: ':',
 11: ';',
 12: '?',
 13: 'A',
 14: 'B',
 15: 'C',
 16: 'D',
 17: 'E',
 18: 'F',
 19: 'G',
 20: 'H',
 21: 'I',
 22: 'J',
 23: 'K',
 24: 'L',
 25: 'M',
 26: 'N',
 27: 'O',
 28: 'P',
 29: 'Q',
 30: 'R',
 31: 'S',
 32: 'T',
 33: 'U',
 34: 'V',
 35: 'W',
 36: 'X',
 37: 'Y',
 38: 'Z',
 39: 'a',
 40: 'b',
 41: 'c',
 42: 'd',
 43: 'e',
 44: 'f',
 45: 'g',
 46: 'h',
 47: 'i',
 48: 'j',
 49: 'k',
 50: 'l',
 51: 'm',
 52: 'n',
 53: 'o',
 54: 'p',
 55: 'q',
 56: 'r',
 57: 's',
 58: 't',
 59: 'u',
 60: 'v',
 61: 'w',
 62: 'x',
 63: 'y',
 64: 'z'}

In [11]:
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: ''.join(itos[i] for i in ids)

In [12]:
encode("Hello")

[20, 43, 50, 50, 53]

In [13]:
decode(encode("Hello"))

'Hello'

In [14]:
data = torch.tensor(encode(text), dtype=torch.long)

n = int(0.9 * len(data))

train_df = data[:n]
val_df = data[n:]

In [15]:
torch.cuda.get_device_name()

'NVIDIA GeForce RTX 3050 Laptop GPU'

In [16]:
device = torch.device("cuda")
device

device(type='cuda')

In [17]:
batch_size = 64
seq_length = 100

def get_batch(data):
    ix = torch.randint(0, len(data) - seq_length, (batch_size,))
    x = torch.stack([data[i:i+seq_length] for i in ix])
    y = torch.stack([data[i+1:i+seq_length+1] for i in ix])
    return x.to(device), y.to(device)

In [18]:
model = RNN(vocab_size=vocab_size).to(device)

Loss = nn.CrossEntropyLoss()

optim = torch.optim.Adam(model.parameters(), lr=0.01)

model

RNN(
  (embedding): Embedding(65, 64)
  (rnn): RNN(64, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=65, bias=True)
)

In [19]:
from tqdm.notebook import tqdm

epochs = 3000

model.train()

progress = tqdm(range(epochs), desc="Tranning RNN")

for epoch in progress:
    X, Y = get_batch(train_df)
    optim.zero_grad()
    logits, hidden = model(X)

    loss = Loss(logits.reshape(-1, vocab_size), Y.reshape(-1))

    loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    optim.step()

    if (epoch + 1) % 50 == 0: progress.set_postfix(loss=f"{loss.item():.2f}")

Tranning RNN:   0%|          | 0/3000 [00:00<?, ?it/s]

In [20]:
model.eval()

val_losses = []

with torch.no_grad():
    for ___ in range(100):
        X, Y = get_batch(val_df)
        logits, hidden = model(X)
        loss = Loss(logits.reshape(-1, vocab_size), Y.reshape(-1))
        val_losses.append(loss.item())

In [21]:
val_loss = sum(val_losses)/len(val_losses)

In [22]:
val_loss

1.7532222938537598

In [24]:
torch.save({"model_state_dict": model.state_dict(), "vocab_size": vocab_size, "stoi": stoi, "itos": itos}, "RNN.pth")

In [29]:
dummy_input = torch.randint(
    0, vocab_size,
    (1, 100),
    dtype=torch.long,
    device=device
)

torch.onnx.export(  model, (dummy_input,), "RNN.onnx", input_names=["input"], output_names=["logits", "hidden"],
                    dynamic_axes={"input": {0: "batch_size", 1: "sequence_length"},"logits": {0: "batch_size", 1: "sequence_length"}, "hidden": {1: "batch_size"}},
                    opset_version=17, dynamo=False                                                                                                                )

/tmp/ipykernel_15606/3710107346.py:8: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(  model, (dummy_input,), "RNN.onnx", input_names=["input"], output_names=["logits", "hidden"],


In [30]:
import os
import onnx

path = "RNN.onnx"

print("File exists:", os.path.exists(path))

model_onnx = onnx.load(path)
onnx.checker.check_model(model_onnx)

print("ONNX model is valid!")
print(f"File size: {os.path.getsize(path) / 1024:.2f} KB")


File exists: True
ONNX model is valid!
File size: 147.94 KB


In [31]:
def generate_text(model, prompt="ROMEO:", length=500, temperature=0.8):
    model.eval()

    if temperature <= 0: raise ValueError("Temperature must be positive")

    generated = prompt
    hidden = None

    with torch.no_grad():
        for char in prompt:
            x = torch.tensor([[stoi[char]]], dtype=torch.long, device=device)
            _, hidden = model.rnn(model.embedding(x), hidden)

        for _ in range(length):
            logits = model.fc(hidden[-1])
            logits = logits / temperature

            probs = torch.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1).item()

            next_char = itos[next_id]
            generated += next_char

            x = torch.tensor([[next_id]], dtype=torch.long, device=device)

            _, hidden = model.rnn(model.embedding(x), hidden)

    return generated


In [35]:
print(generate_text(model, "What is up homie", length=20))

What is up homies and harries;
'Twas
